# 02. Fit rungs 0 to 2 to the basic trials

**What this notebook does.** It teaches three kinds of simulated user from the **basic (non-cued) trials only**. Cued trials are never used here.

| Rung | Simulated user | What it learns per person |
| --- | --- | --- |
| 0 | Simple statistics | How reward size and delay move the chance of choosing "later" |
| 1 | Patience | A patience level and how consistent the choices are |
| 2 | Patience + decision time | Patience, caution, and timing traits; predicts how long each choice takes |

Every model is **hierarchical Bayesian**: each person's traits are estimated together with the group's, so a person with noisy data is pulled gently towards the group average instead of getting an extreme estimate. Each person has traits for day 1 and day 2, tied together by a personal average.

Rung 3 (practice and attention states) is added in a later notebook.

In [1]:
import sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, "../src")
import dd

# QUICK = True runs a 5-minute version on 8 people to check that everything works.
# QUICK = False is the real analysis (rung 2 takes the longest; see notebook 02a for a time estimate).
QUICK = False
N_PEOPLE = 8 if QUICK else None
CHAINS, WARMUP, SAMPLES = (2, 150, 150) if QUICK else (4, 1000, 1000)
TAG = "quick" if QUICK else "basic"
RUNGS = ["rung0", "rung1", "rung2"]


In [2]:
d = dd.reindex(dd.load_trials(), N_PEOPLE)
basic = d[d.condition == "basic"].reset_index(drop=True)
print(f"{len(basic):,} basic trials · {basic.p.nunique()} people · {basic.u.nunique()} person-sessions")

8,004 basic trials · 42 people · 84 person-sessions


## Fit
For each rung: fit, record three health numbers, save the estimates.

- **max R-hat**: do the separate runs agree? Should be below 1.01.
- **min ESS**: how many independent samples the estimate is worth. Should be above 400.
- **divergent**: warnings that the sampler struggled. Should be 0.

In [3]:
rows = []
for rung in RUNGS:
    t0 = time.time()
    f = dd.fit(rung, dd.stan_data(basic, with_rt=(rung == "rung2")), chains=CHAINS, warmup=WARMUP, samples=SAMPLES)
    dd.save_draws(f, rung, TAG)
    rows.append({"rung": rung, **dd.diagnostics(f), "minutes": round((time.time() - t0) / 60, 1)})
    print(rows[-1])
diag = pd.DataFrame(rows).set_index("rung")
diag["passed"] = (diag.max_rhat < 1.01) & (diag.divergent == 0) & (diag.min_ess >= 400)
diag.to_csv(dd.FIT_DIR / f"diagnostics_{TAG}.csv")
diag

18:32:11 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

18:33:10 - cmdstanpy - INFO - CmdStan done processing.
18:33:10 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'rung0_logistic.stan', line 31, column 17 to column 58)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'rung0_logistic.stan', line 31, column 17 to column 58)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'rung0_logistic.stan', line 31, column 17 to column 58)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'rung0_logistic.stan', line 31, column 17 to column 58)
Consider re-running with show_console=True if the above output is unclear!


... (12 outputs left out)

chain 4:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

19:25:09 - cmdstanpy - INFO - CmdStan done processing.
19:25:09 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: wiener_lpdf: Boundary separation[1] is inf, but must be positive finite! (in 'rung2_ddm.stan', line 63, column 2 to column 73)
	Exception: wiener_lpdf: Boundary separation[1] is inf, but must be positive finite! (in 'rung2_ddm.stan', line 63, column 2 to column 73)
	Exception: wiener_lpdf: Boundary separation[1] is inf, but must be positive finite! (in 'rung2_ddm.stan', line 63, column 2 to column 73)
	Exception: wiener_lpdf: Boundary separation[1] is 0, but must be positive finite! (in 'rung2_ddm.stan', line 63, column 2 to column 73)
	Exception: wiener_lpdf: Boundary separation[1] is 0, but must be positive finite! (in 'rung2_ddm.stan', line 63, column 2 to column 73)
Exception: wiener_lpdf: Boundary separation[1] is inf, but must be positive finite! (in 'rung2_ddm.stan', line 63, column 2 to column 73)
	Exception: wiener_lpdf: Boundary separation[1] is 


{'rung': 'rung2', 'max_rhat': 1.005, 'min_ess': 541, 'divergent': 0, 'max_treedepth_share': 0.0, 'minutes': 50.9}


,max_rhat,min_ess,divergent,max_treedepth_share,minutes,passed
rung,,,,,,
rung0,1.005,481,0,0.0,1.0,True
rung1,1.004,1535,0,0.0,1.2,True
rung2,1.005,541,0,0.0,50.9,True


## What the group looks like (rung 2)
Group averages, converted to everyday units, with the range that covers 90% of the uncertainty.

In [6]:
dr = dd.load_draws("rung2", TAG); mu = dr["mu"]
conv = {"Patience: discount rate k (per day; lower = more patient)": np.exp(mu[:, 0]),
        "Caution: boundary separation": np.exp(mu[:, 1]),
        "Starting lean towards 'later' (0.5 = none)": 1 / (1 + np.exp(-mu[:, 3])),
        "Maximum drift": np.exp(mu[:, 4]),
        "Drift sensitivity to value": np.exp(mu[:, 5])}
pd.DataFrame({k: [np.median(v), np.quantile(v, .05), np.quantile(v, .95)] for k, v in conv.items()},
             index=["typical person", "low", "high"]).T.round(4)

,typical person,low,high
Patience: discount rate k (per day; lower = more patient),0.0047,0.0032,0.0071
Caution: boundary separation,2.7631,2.6270,2.9052
Starting lean towards 'later' (0.5 = none),0.5603,0.5420,0.5788
Maximum drift,1.2686,1.1644,1.3834
Drift sensitivity to value,0.3320,0.2748,0.4011


## Save each person's traits
One row per person and session, per rung: the average estimate and its uncertainty. Later notebooks read this file.

In [7]:
out = []
units = d.groupby("u")[["person", "session"]].first()
for rung in RUNGS:
    th = dd.load_draws(rung, TAG)["theta"]                  # draws x units x traits
    for k, trait in enumerate(dd.TRAITS[rung]):
        out.append(pd.DataFrame({"rung": rung, "trait": trait, "person": units.person.values, "session": units.session.values,
                                 "mean": th[:, :, k].mean(0), "sd": th[:, :, k].std(0)}))
traits = pd.concat(out, ignore_index=True)
traits.to_csv(dd.FIT_DIR / f"traits_{TAG}.csv", index=False)
traits[traits.rung == "rung2"].pivot_table(index=["person", "session"], columns="trait", values="mean").head(6).round(2)

trait           log_alpha  log_vcoef  log_vmax  logit_bias  logit_tau  logk
person session                                                             
1      1             1.29      -0.84     -0.11        0.27       1.78 -4.19
       2             1.04      -0.27     -0.01        0.25       1.74 -5.19
2      1             0.92      -1.38      0.15       -0.22       1.97 -2.79
       2             0.87      -1.75      0.29       -0.07       1.93 -3.01
3      1             1.28      -0.75     -0.16        0.27       1.70 -5.14
       2             1.05      -0.54      0.06        0.33       1.74 -4.62